# Publication notebook: tuned-vs-baseline classical comparison

- Purpose: compare retained tuned and baseline classical models and derive feature-ablation outputs.
- Required inputs: pair-noq processed TSVs and retained classical metrics under `results/metrics/classical/`.
- Required models: tuned classical `.joblib` artifacts in `models/pair_noq_tuned/`.
- Required external tools: none.
- Expected outputs: ablation JSON/TSV files and comparison figures under `results/`.
- Publication output: tuned-vs-baseline tables, permutation importance, and feature-ablation summaries.
- Reproduction status: normalized to canonical paths, but a fresh clone still needs local tuned model binaries.


In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.inspection import permutation_importance
from sklearn.base import clone

In [ ]:
def load_pair_noq_like_hparam_script(path: str):
    """
    Match mitochime.hyperparam_search_top.load_dataset()
    """
    from sklearn.impute import SimpleImputer

    df = pd.read_csv(path, sep="\t")
    if "label" not in df.columns:
        raise ValueError("Expected a 'label' column in the dataset.")

    if "strand" in df.columns:
        df["strand"] = df["strand"].map({"+": 1, "-": 0})

    drop_cols = ["read_id", "ref_name", "cigar"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    for c in df.columns:
        if c != "label":
            df[c] = pd.to_numeric(df[c], errors="coerce")

    feature_cols = [c for c in df.columns if c != "label"]
    X = df[feature_cols].to_numpy(dtype=float)
    y = df["label"].to_numpy(dtype=int)

    imp = SimpleImputer(strategy="median")
    X = imp.fit_transform(X)

    return X, y, feature_cols

In [ ]:
baseline_path = Path("../results/metrics/classical/baseline_pair_noq/metrics_summary.tsv")
tuned_path = Path("../results/metrics/classical/tuned_pair_noq/tuned_models_summary.tsv")

baseline = pd.read_csv(baseline_path, sep="\t")
tuned = pd.read_csv(tuned_path, sep="\t")

print("Baseline shape:", baseline.shape)
print("Tuned shape:", tuned.shape)
baseline.head(), tuned.head()

In [ ]:
for df_ in (baseline, tuned):
    if "model_base" in df_.columns:
        del df_["model_base"]

tuned["model_base"] = tuned["model"].str.replace("_tuned$", "", regex=True)
baseline["model_base"] = baseline["model"]

for df_ in (baseline, tuned):
    if "model" in df_.columns:
        del df_["model"]

merged = baseline.merge(
    tuned,
    on="model_base",
    suffixes=("_base", "_tuned"),
)

merged["model_name"] = merged["model_base"]
merged = merged.sort_values("test_f1_tuned", ascending=False).reset_index(drop=True)

merged[[
    "model_name",
    "test_f1_base",
    "test_f1_tuned",
    "test_roc_auc_base",
    "test_roc_auc_tuned",
]]

In [ ]:
plot_df = merged.copy()
x = np.arange(len(plot_df))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

axes[0].bar(x - 0.2, plot_df["test_f1_base"], width=0.4, label="baseline")
axes[0].bar(x + 0.2, plot_df["test_f1_tuned"], width=0.4, alpha=0.8, label="tuned")
axes[0].set_xticks(x)
axes[0].set_xticklabels(plot_df["model_name"], rotation=45, ha="right")
axes[0].set_ylabel("Test F1")
axes[0].set_title("F1 before vs after tuning")
axes[0].legend()

axes[1].bar(x - 0.2, plot_df["test_roc_auc_base"], width=0.4, label="baseline")
axes[1].bar(x + 0.2, plot_df["test_roc_auc_tuned"], width=0.4, alpha=0.8, label="tuned")
axes[1].set_xticks(x)
axes[1].set_xticklabels(plot_df["model_name"], rotation=45, ha="right")
axes[1].set_ylabel("Test ROC-AUC")
axes[1].set_title("ROC-AUC before vs after tuning")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
X_test, y_test, feature_names = load_pair_noq_like_hparam_script("../data/processed/PAIR_test_noq.tsv")

print("X_test:", X_test.shape)
print("n_features:", len(feature_names))
print("first 10 features:", feature_names[:10])

models = {
    "gradient_boosting": joblib.load("../models/pair_noq_tuned/gradient_boosting_tuned.joblib"),
    "catboost": joblib.load("../models/pair_noq_tuned/catboost_tuned.joblib"),
    "lightgbm": joblib.load("../models/pair_noq_tuned/lightgbm_tuned.joblib"),
    "bagging_trees": joblib.load("../models/pair_noq_tuned/bagging_trees_tuned.joblib"),
    "mlp": joblib.load("../models/pair_noq_tuned/mlp_tuned.joblib"),
}

In [ ]:
def plot_cm(cm, title):
    plt.figure(figsize=(4.5, 4.0))
    plt.imshow(cm)
    plt.title(title)
    plt.xticks([0, 1], ["clean", "chimeric"])
    plt.yticks([0, 1], ["clean", "chimeric"])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    for (i, j), v in np.ndenumerate(cm):
        plt.text(j, i, str(v), ha="center", va="center")
    plt.tight_layout()
    plt.show()

for name, model in models.items():
    print(f"\n=== {name} ===")
    y_pred = model.predict(X_test)
    y_pred = np.asarray(y_pred).reshape(-1).astype(int)

    print(classification_report(
        y_test,
        y_pred,
        target_names=["clean", "chimeric"],
        digits=4,
        zero_division=0,
    ))

    cm = confusion_matrix(y_test, y_pred)
    plot_cm(cm, f"Confusion matrix – {name} (PAIR_noq)")

In [ ]:
gb = models["gradient_boosting"]

proba = gb.predict_proba(X_test)[:, 1]

best_t, best_f1 = 0.5, 0.0
for t in [i / 100 for i in range(1, 100)]:
    pred = (proba >= t).astype(int)
    f1 = f1_score(y_test, pred, zero_division=0)
    if f1 > best_f1:
        best_f1, best_t = f1, t

print("best_t:", best_t)
print("best_f1:", best_f1)

In [ ]:
out_path = Path("../models/pair_noq_tuned/feature_cols_24.json")
out_path.write_text(json.dumps(feature_names, indent=2) + "\n")
print("Saved:", out_path.resolve())
print("n_features:", len(feature_names))

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
for name, model in models.items():
    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
    else:
        continue

    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", lw=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves")
plt.legend()

plt.subplot(1, 2, 2)
for name, model in models.items():
    if hasattr(model, "predict_proba"):
        scores = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
    else:
        continue

    prec, rec, _ = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)
    plt.plot(rec, prec, label=f"{name} (AP={ap:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curves")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
def perm_importance(model, X, y, feature_names, scoring="f1", n_repeats=10):
    r = permutation_importance(
        model,
        X,
        y,
        n_repeats=n_repeats,
        random_state=42,
        n_jobs=-1,
        scoring=scoring,
    )
    df_imp = pd.DataFrame({
        "feature": feature_names,
        "importance_mean": r.importances_mean,
        "importance_std": r.importances_std,
    }).sort_values("importance_mean", ascending=False).reset_index(drop=True)
    return df_imp

imp_results = {}

for name, model in models.items():
    print(f"\n=== Permutation importance ({name}) ===")
    df_imp = perm_importance(model, X_test, y_test, feature_names, scoring="f1", n_repeats=10)
    imp_results[name] = df_imp
    display(df_imp.head(10))

In [ ]:
def plot_top10(df_imp, title):
    top = df_imp.head(10).iloc[::-1]
    plt.figure(figsize=(7, 4))
    plt.barh(top["feature"], top["importance_mean"])
    plt.xlabel("Permutation importance (ΔF1)")
    plt.title(title)
    plt.tight_layout()
    plt.show()

for name in ["gradient_boosting", "catboost", "bagging_trees", "mlp"]:
    plot_top10(imp_results[name], f"Top 10 features – {name}")

In [ ]:
family_map = {
    "SA_structure": [
        "has_sa",
        "sa_count",
        "num_segments",
        "sa_diff_contig",
        "sa_min_delta_pos",
        "sa_max_delta_pos",
        "sa_mean_delta_pos",
        "sa_same_strand_count",
        "sa_opp_strand_count",
        "sa_max_mapq",
        "sa_mean_mapq",
        "sa_min_nm",
        "sa_mean_nm",
    ],
    "Clipping": [
        "softclip_left",
        "softclip_right",
        "total_clipped_bases",
        "breakpoint_read_pos",
    ],
    "Kmer_jump": [
        "kmer_cosine_diff",
        "kmer_js_divergence",
    ],
    "Micro_homology": [
        "microhomology_length",
        "microhomology_gc",
    ],
    "Other": [
        "mapq",
        "read_length",
        "strand",
    ],
}

In [ ]:
def aggregate_family_importance(df_imp: pd.DataFrame, family_map: dict) -> pd.DataFrame:
    rows = []
    for fam, feats in family_map.items():
        sub = df_imp[df_imp["feature"].isin(feats)]
        if sub.empty:
            continue
        rows.append({
            "family": fam,
            "importance_sum": sub["importance_mean"].sum(),
            "importance_mean": sub["importance_mean"].mean(),
            "n_features": len(sub),
        })
    fam_df = pd.DataFrame(rows).sort_values("importance_sum", ascending=False)
    return fam_df

family_results = {}

for model_name, df_imp in imp_results.items():
    fam_df = aggregate_family_importance(df_imp, family_map)
    family_results[model_name] = fam_df
    print(f"\n=== Aggregated feature families – {model_name} ===")
    display(fam_df)

In [ ]:
for model_name, fam_df in family_results.items():
    fam_sorted = fam_df.sort_values("importance_sum", ascending=True)
    plt.figure(figsize=(6, 4))
    plt.barh(fam_sorted["family"], fam_sorted["importance_sum"])
    plt.xlabel("Total permutation importance (sum ΔF1)")
    plt.title(f"Feature family importance – {model_name}")
    plt.tight_layout()
    plt.show()

In [ ]:
train_path = "../data/processed/PAIR_train_noq.tsv"
test_path = "../data/processed/PAIR_test_noq.tsv"
model_path = "../models/pair_noq_tuned/gradient_boosting_tuned.joblib"

out_dir = Path("../results/metrics/classical/gradient_boosting_analysis_pair_noq")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "gradient_boosting_perm_importance.tsv"

X_train, y_train, feature_names_train = load_pair_noq_like_hparam_script(train_path)
X_test, y_test, feature_names_test = load_pair_noq_like_hparam_script(test_path)

assert feature_names_train == feature_names_test, "Feature schema mismatch between train and test."

gb_model = joblib.load(model_path)

result = permutation_importance(
    gb_model,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
    scoring="f1",
)

gb_imp = pd.DataFrame({
    "feature": feature_names_train,
    "importance_mean": result.importances_mean,
    "importance_std": result.importances_std,
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

print(gb_imp.head(15))
gb_imp.to_csv(out_path, sep="\t", index=False)
print("Wrote:", out_path.resolve())

In [ ]:
gb_model = joblib.load("../models/pair_noq_tuned/gradient_boosting_tuned.joblib")
type(gb_model), gb_model.get_params()

In [ ]:
gb_imp = gb_imp.copy()
gb_imp["importance_pos"] = gb_imp["importance_mean"].clip(lower=0)
total_pos = float(gb_imp["importance_pos"].sum())

print("Total positive importance:", total_pos)

if total_pos == 0.0:
    gb_imp["importance_frac"] = 0.0
    gb_imp["cum_frac"] = 0.0
else:
    gb_imp = gb_imp.sort_values("importance_pos", ascending=False).reset_index(drop=True)
    gb_imp["importance_frac"] = gb_imp["importance_pos"] / total_pos
    gb_imp["cum_frac"] = gb_imp["importance_frac"].cumsum()

gb_imp.head(10)

In [ ]:
threshold = 0.95

gb_imp = gb_imp.copy()
gb_imp["importance_pos"] = gb_imp["importance_mean"].clip(lower=0)
total_pos = float(gb_imp["importance_pos"].sum())

if total_pos == 0.0:
    raise ValueError("Total positive importance is 0. Nothing to threshold-select.")

gb_imp = gb_imp.sort_values("importance_pos", ascending=False).reset_index(drop=True)
gb_imp["importance_frac"] = gb_imp["importance_pos"] / total_pos
gb_imp["cum_frac"] = gb_imp["importance_frac"].cumsum()

k = int((gb_imp["cum_frac"] < threshold).sum()) + 1
selected_features = gb_imp.iloc[:k]["feature"].tolist()

print(f"Selected {len(selected_features)} features to reach >= {threshold*100:.0f}% cumulative positive importance")
print("Top selected features:")
for f in selected_features:
    print("-", f)

print("Reached cum_frac:", float(gb_imp.iloc[k - 1]["cum_frac"]))

In [ ]:
out_dir = Path("../results/metrics/classical/gradient_boosting_analysis_pair_noq")
out_dir.mkdir(parents=True, exist_ok=True)

txt_path = out_dir / "selected_features_95pct.txt"
json_path = out_dir / "selected_features_95pct.json"

txt_path.write_text("\n".join(selected_features) + "\n")
json_path.write_text(json.dumps(selected_features, indent=2) + "\n")

print("Saved:")
print("-", txt_path.resolve())
print("-", json_path.resolve())

In [ ]:
def get_pos_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        s = np.asarray(s)
        if s.ndim > 1:
            s = s[:, 1]
        return s
    return model.predict(X)

def eval_sklearn_on(Xtr, Xte, ytr, yte, base_model):
    m = clone(base_model)
    m.fit(Xtr, ytr)
    y_pred = m.predict(Xte)
    y_pred = np.asarray(y_pred).reshape(-1).astype(int)

    scores = get_pos_scores(m, Xte)
    scores = np.asarray(scores).reshape(-1)

    f1 = float(f1_score(yte, y_pred, zero_division=0))
    roc = float(roc_auc_score(yte, scores))
    return f1, roc

X_train, y_train, feature_names = load_pair_noq_like_hparam_script(train_path)
X_test, y_test, feature_names_test = load_pair_noq_like_hparam_script(test_path)

assert feature_names_test == feature_names

name_to_idx = {f: i for i, f in enumerate(feature_names)}

def cols_to_idx(cols):
    missing = [c for c in cols if c not in name_to_idx]
    if missing:
        raise ValueError(f"Missing features: {missing}")
    return [name_to_idx[c] for c in cols]

selected_idx = cols_to_idx(selected_features)

micro_features = ["microhomology_length", "microhomology_gc"]
nomicro_idx = [name_to_idx[f] for f in feature_names if f not in micro_features]

Xtr_full, Xte_full = X_train, X_test
Xtr_sel, Xte_sel = X_train[:, selected_idx], X_test[:, selected_idx]
Xtr_nom, Xte_nom = X_train[:, nomicro_idx], X_test[:, nomicro_idx]

f1_full, roc_full = eval_sklearn_on(Xtr_full, Xte_full, y_train, y_test, gb_model)
print(f"Full model: {Xtr_full.shape[1]} features | F1={f1_full:.4f} | ROC-AUC={roc_full:.4f}")

f1_sel, roc_sel = eval_sklearn_on(Xtr_sel, Xte_sel, y_train, y_test, gb_model)
print(f"Selected-feat model: {Xtr_sel.shape[1]} features | F1={f1_sel:.4f} | ROC-AUC={roc_sel:.4f}")

f1_nom, roc_nom = eval_sklearn_on(Xtr_nom, Xte_nom, y_train, y_test, gb_model)
print(f"No-micro model: {Xtr_nom.shape[1]} features | F1={f1_nom:.4f} | ROC-AUC={roc_nom:.4f}")

In [ ]:
results = {
    "paths": {
        "train_tsv": train_path,
        "test_tsv": test_path,
        "model_path": model_path,
    },
    "full": {
        "n_features": int(Xtr_full.shape[1]),
        "f1": f1_full,
        "roc_auc": roc_full,
        "features": feature_names,
    },
    "selected": {
        "n_features": int(Xtr_sel.shape[1]),
        "f1": f1_sel,
        "roc_auc": roc_sel,
        "features": selected_features,
    },
    "no_microhomology": {
        "n_features": int(Xtr_nom.shape[1]),
        "f1": f1_nom,
        "roc_auc": roc_nom,
        "dropped_features": micro_features,
        "kept_features": [f for f in feature_names if f not in micro_features],
    },
}

out_json = Path("../results/metrics/classical/gradient_boosting_analysis_pair_noq/gradient_boosting_feature_ablation_results.json")
out_json.write_text(json.dumps(results, indent=2) + "\n")

print("Saved:", out_json.resolve())

In [ ]:
gb_plot = gb_imp.copy()
gb_plot["importance_pos"] = gb_plot["importance_mean"].clip(lower=0)
total_pos = float(gb_plot["importance_pos"].sum())

if total_pos == 0:
    raise ValueError("Total positive importance is 0.")

gb_plot = gb_plot.sort_values("importance_pos", ascending=False).reset_index(drop=True)
gb_plot["importance_frac"] = gb_plot["importance_pos"] / total_pos
gb_plot["cum_frac"] = gb_plot["importance_frac"].cumsum()

n_features = len(gb_plot)
x = np.arange(1, n_features + 1)
cum_importance = gb_plot["cum_frac"].to_numpy()

threshold = 0.95
k_thresh = int(np.searchsorted(cum_importance, threshold, side="left") + 1)

plt.figure(figsize=(6, 4))
plt.plot(x, cum_importance, marker="o")
plt.axhline(threshold, linestyle="--")
plt.axvline(k_thresh, linestyle="--")
plt.xlabel("Number of features (sorted by importance)")
plt.ylabel("Cumulative positive permutation importance")
plt.title("Cumulative importance (GradientBoosting, permutation ΔF1)")
plt.tight_layout()
plt.show()

print(f"{threshold*100:.0f}% cumulative importance reached at k = {k_thresh}")
print("Top features up to threshold:")
for f in gb_plot.loc[:k_thresh - 1, "feature"].tolist():
    print("-", f)

In [ ]:
model_name = "GradientBoosting"

variants = [
    ("Full", "full"),
    ("Selected subset", "selected"),
    ("No microhomology", "no_microhomology"),
]

f1_scores = [float(results[k]["f1"]) for _, k in variants]
auc_scores = [float(results[k]["roc_auc"]) for _, k in variants]
labels = [f"{name}\n(n={int(results[k]['n_features'])})" for name, k in variants]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(7, 4))
plt.bar(x - width / 2, f1_scores, width, label="F1")
plt.bar(x + width / 2, auc_scores, width, label="ROC-AUC")
plt.xticks(x, labels)
plt.ylabel("Score")
plt.ylim(0.0, 1.0)
plt.title(f"{model_name} performance under different feature sets")
plt.legend()
plt.tight_layout()
plt.show()

for name, k in variants:
    print(
        f"{name:16s} n_features={int(results[k]['n_features']):>2d} "
        f"F1={float(results[k]['f1']):.4f} ROC-AUC={float(results[k]['roc_auc']):.4f}"
    )

In [ ]:
out_dir = Path("../results/metrics/classical/gradient_boosting_analysis_pair_noq")
out_dir.mkdir(parents=True, exist_ok=True)

perm_imp_df = gb_imp
prefix = "gradient_boosting"

rows = []
for key in ["full", "selected", "no_microhomology"]:
    rows.append({
        "variant": key,
        "n_features": int(results[key]["n_features"]),
        "f1": float(results[key]["f1"]),
        "roc_auc": float(results[key]["roc_auc"]),
    })

df_fs = pd.DataFrame(rows)

metrics_path = out_dir / f"{prefix}_feature_selection_metrics_pair.tsv"
df_fs.to_csv(metrics_path, sep="\t", index=False)

sel_path = out_dir / f"{prefix}_selected_features_95pct_pair.txt"
with open(sel_path, "w") as f:
    for feat in selected_features:
        f.write(str(feat) + "\n")

perm_path = out_dir / f"{prefix}_perm_importance_pair.tsv"
perm_imp_df.to_csv(perm_path, sep="\t", index=False)

print("Saved to:", out_dir.resolve())
print("Wrote metrics:", metrics_path.name)
print("Wrote selected features:", sel_path.name)
print("Wrote perm importance:", perm_path.name)